In [1]:
!pip install imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 61.4 MB/s eta 0:00:00


In [2]:
import os
import cv2
import time
import re
import numpy as np
from werkzeug.utils import secure_filename
from collections import Counter
import cv2
import imagehash
from PIL import Image
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

In [3]:
def FrameCapture(video_path):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    video_folder = os.path.join(video_name)
    os.makedirs(video_folder, exist_ok=True)
    vidObj = cv2.VideoCapture(video_path)
    count = 0
    success = True
    while success:
        success, image = vidObj.read()
        if success:
            frame_path = os.path.join(video_folder, f"frame{count}.png")
            cv2.imwrite(frame_path, image, [cv2.IMWRITE_PNG_COMPRESSION, 0])
            count += 1
    return video_folder


In [4]:
def natural_sort_key(filename):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', filename)]
def histogram_difference(frame1, frame2):
    hist1 = cv2.calcHist([frame1], [0], None, [256], [0, 256])
    hist2 = cv2.calcHist([frame2], [0], None, [256], [0, 256])
    diff = cv2.compareHist(hist1, hist2, cv2.HISTCMP_BHATTACHARYYA)
    return diff

In [5]:
def compute_histogram_differences(frame_folder):
    files = sorted([f for f in os.listdir(frame_folder) if f.endswith('.png')], key=natural_sort_key)
    diffs = []

    prev_frame = cv2.imread(os.path.join(frame_folder, files[0]))
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    for i in range(1, len(files)):
        curr_frame = cv2.imread(os.path.join(frame_folder, files[i]))
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)

        diff = histogram_difference(prev_gray, curr_gray)
        diffs.append(diff)

        prev_gray = curr_gray

    return diffs, files

def scene_change_detection(frame_folder, k=1.5, min_distance=5):
    diffs, files = compute_histogram_differences(frame_folder)
    mean_diff = np.mean(diffs)
    std_diff = np.std(diffs)

    adaptive_threshold = mean_diff + k * std_diff
    #print(f"Adaptive Histogram threshold: {adaptive_threshold:.4f} (mean={mean_diff:.4f}, std={std_diff:.4f})")

    scene_changes = [files[0]]
    last_change_index = 0

    for i, diff in enumerate(diffs):
        if diff > adaptive_threshold:
            if (i + 1) - last_change_index >= min_distance:
                scene_changes.append(files[i + 1])
                last_change_index = i + 1

    return scene_changes, adaptive_threshold, len(scene_changes)


In [6]:


def otsu_threshold(image):
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    threshold_value, _ = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return threshold_value

def edge_detection(frame):
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred_image = cv2.GaussianBlur(gray_frame, (5, 5), 0)
    otsu_threshold_v = otsu_threshold(blurred_image)
    lower_threshold = 0.5 * otsu_threshold_v
    upper_threshold = otsu_threshold_v
    edges = cv2.Canny(blurred_image, int(lower_threshold), int(upper_threshold))
    edges = (edges > np.percentile(edges, 95)).astype(np.uint8) * 255
    return edges


def divide_into_blocks(frame, block_size):
    height, width = frame.shape
    blocks, positions = [], []
    for i in range(0, height - block_size + 1, block_size):
        for j in range(0, width - block_size + 1, block_size):
            block = frame[i:i + block_size, j:j + block_size]
            blocks.append(block)
            positions.append((i, j))

    return blocks, positions



def d_embedding_error(block, watermark):
    return np.sum(np.abs(block.astype(int) - watermark.astype(int)))


def d_edge_density(block):
    return np.count_nonzero(block) / block.size




def best_block(frame, watermark):
    block_size = watermark.shape[0]
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = edge_detection(frame)

    frame_blocks, positions = divide_into_blocks(gray_frame, block_size)
    edge_blocks, _ = divide_into_blocks(edges, block_size)

    max_edge_density, best_block_index, min_error = 0, -1, float('inf')

    for idx, block in enumerate(frame_blocks):
        edge_density = d_edge_density(edge_blocks[idx])
        if edge_density > max_edge_density:
            error = d_embedding_error(block, watermark)
            if error < min_error:
                max_edge_density, min_error, best_block_index = edge_density, error, idx

    return positions[best_block_index]





def text_to_bin(data):
    return ''.join([format(ord(i), '08b') for i in data])

def bin_to_text(binary_data):
    all_bytes = [binary_data[i:i+8] for i in range(0, len(binary_data), 8)]
    return ''.join([chr(int(byte, 2)) for byte in all_bytes])


In [7]:
def KSA(key):
    key_length = len(key)
    S=list(range(256))
    j=0
    for i in range(256):
        j=(j+S[i]+key[i % key_length]) % 256
        S[i],S[j]=S[j],S[i]
    return S

def PRGA(S,n):
    i=0
    j=0
    key=[]
    while n>0:
        n=n-1
        i=(i+1)%256
        j=(j+S[i])%256
        S[i],S[j]=S[j],S[i]
        K=S[(S[i]+S[j])%256]
        key.append(K)
    return key

def preparing_key_array(s):
    return [ord(c) for c in s]

def encryption(plaintext,key):
    #print("Enter the key : ") adding this as parameter
    #key=input()
    key=preparing_key_array(key)

    S=KSA(key)

    keystream=np.array(PRGA(S,len(plaintext)))
    plaintext=np.array([ord(i) for i in plaintext])

    cipher=keystream^plaintext
    ctext=''
    for c in cipher:
        ctext=ctext+chr(c)
    return ctext

def decryption(ciphertext,key):
    #print("Enter the key : ") adding ths as the parameter to this function
    #key=input()
    key=preparing_key_array(key)

    S=KSA(key)

    keystream=np.array(PRGA(S,len(ciphertext)))
    ciphertext=np.array([ord(i) for i in ciphertext])

    decoded=keystream^ciphertext
    dtext=''
    for c in decoded:
        dtext=dtext+chr(c)
    return dtext







In [8]:
def encode(img, data,key):
    img_data = np.array(img, dtype=np.uint8)
    if img_data.shape[2] == 4:
        img_data = img_data[:, :, :3]
    delimiter = "###END###"
    data=encryption(data,key)
    binary_data = text_to_bin(data + delimiter)
    length_bin = format(len(binary_data), '032b')
    binary_data = length_bin + binary_data
    idx, data_len = 0, len(binary_data)
    x, y = best_block(img_data, np.zeros((32, 32)))
    block_size = 32
    for i in range(block_size):
        for j in range(block_size):
            for channel in range(3):
                if idx < data_len:
                        current_pixel = img_data[x + i, y + j, channel]
                        bit_to_embed = int(binary_data[idx])
                        if bit_to_embed not in [0, 1]:
                            raise ValueError(f"Invalid bit value: {bit_to_embed}")
                        modified_pixel = np.uint8((current_pixel & ~np.uint8(1)) | np.uint8(bit_to_embed))
                        img_data[x + i, y + j, channel] = modified_pixel
                        idx += 1
    return img_data

In [9]:
def decode(img,key):
    img_data = np.array(img)

    if img_data.shape[2] == 4:
        img_data = img_data[:, :, :3]

    binary_data = ''

    x, y = best_block(img_data, np.zeros((32, 32)))
    if x + 32 > img_data.shape[0] or y + 32 > img_data.shape[1]:
        raise ValueError("Block position out of image bounds")

    block_size = 32

    for i in range(block_size):
        for j in range(block_size):
            for channel in range(3):
                binary_data += str(img_data[x + i, y + j, channel] & 1)

    message_length = int(binary_data[:32], 2)
    binary_data = binary_data[32: 32 + message_length]

    # Convert binary to text
    decoded_text = bin_to_text(binary_data)

    # Stop at the delimiter
    if "###END###" in decoded_text:
        decoded_text = decoded_text.split("###END###")[0]
    decoded_text=decryption(decoded_text,key)

    #print(f"Decoding Block Position: {x}, {y}")
    #print(f"Extracted length: {len(decoded_text)}")
    return decoded_text

In [10]:
def calculate_mse(image1, image2):
    err = np.mean((image1 - image2) ** 2)
    return err
def calculate_psnr(image1, image2):
    return cv2.PSNR(image1, image2)
def calculate_ssim(image1, image2):

    if len(image1.shape) == 3:
        image1 = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)
    if len(image2.shape) == 3:
        image2 = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)

    # Compute SSIM
    score, _ = ssim(image1, image2, full=True)
    return score

In [11]:


def extract_frame_number(frame_name):
    match = re.search(r'frame(\d+)', frame_name)
    return int(match.group(1)) if match else -1

def add_Watermark(watermark, frames_folder,key,plot_graph=False):
    encoded_folder = os.path.join(frames_folder, "encoded_frames")
    os.makedirs(encoded_folder, exist_ok=True)

    scenes, _, _ = scene_change_detection(frames_folder,min_distance=3)

    psnr_list = []
    ssim_list = []
    mse_list = []
    frame_numbers = []

    # Step 1: Watermark and save encoded frames
    for frame_name in scenes:
        frame_path = os.path.join(frames_folder, frame_name)
        encoded_frame_path = os.path.join(encoded_folder, frame_name)

        frame = cv2.imread(frame_path)
        if frame is None:
            print(f"Error reading frame: {frame_name}")
            continue

        encoded_frame = encode(frame, watermark,key)
        cv2.imwrite(encoded_frame_path, encoded_frame, [cv2.IMWRITE_PNG_COMPRESSION, 0])

    # Step 2: Compare and collect metrics
    for frame_name in scenes:
        original_path = os.path.join(frames_folder, frame_name)
        encoded_path = os.path.join(encoded_folder, frame_name)

        frame_orig = cv2.imread(original_path)
        frame_encoded = cv2.imread(encoded_path)

        if frame_orig is None or frame_encoded is None:
            continue

        mse = calculate_mse(frame_orig, frame_encoded)
        psnr = calculate_psnr(frame_orig, frame_encoded)
        ssim = calculate_ssim(frame_orig, frame_encoded)

        mse_list.append(mse)
        psnr_list.append(psnr)
        ssim_list.append(ssim)

        frame_number = extract_frame_number(frame_name)
        frame_numbers.append(frame_number)

        # Overwrite original with watermarked version
        os.replace(encoded_path, original_path)

    # Cleanup
    os.rmdir(encoded_folder)

    # Display average metrics
    print("Average PSNR:", np.mean(psnr_list))
    print("Average SSIM:", np.mean(ssim_list))
    print("Average MSE :", np.mean(mse_list))

    # Plot PSNR vs Frame Number
    if plot_graph:
        plt.figure(figsize=(10, 5))
        plt.plot(frame_numbers, psnr_list, marker='o', linestyle='-', color='blue')
        plt.title('PSNR vs Frame Number')
        plt.xlabel('Frame Number')
        plt.ylabel('PSNR (dB)')
        plt.grid(True)
        plt.tight_layout()
        plt.show()
    print("watermark embedded")
    return psnr_list,frame_numbers


In [12]:
def reconstruction(frames_folder, output_path, fps=30):
    frames = sorted([f for f in os.listdir(frames_folder) if f.endswith('.png')], key=natural_sort_key)
    if not frames:
        raise ValueError("No frames found to reconstruct the video.")

    sample_frame = cv2.imread(os.path.join(frames_folder, frames[0]))
    height, width, layers = sample_frame.shape

    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    for frame_name in frames:
        frame = cv2.imread(os.path.join(frames_folder, frame_name))
        out.write(frame)

    out.release()
    return output_path


In [13]:
import os
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from collections import Counter
def extract(folder_path,key) :


  dec = []
  scenes, _, _ = scene_change_detection(folder_path,min_distance=3)
  for frame_name in scenes:
      frame_path = os.path.join(folder_path, frame_name)
      frame = cv2.imread(frame_path)

      if frame is not None:
          decoded_text = decode(frame,key)
          dec.append(decoded_text)
          #print(f"Decoded text from {frame_name}: {decoded_text}")
      else:
          print(f"Failed to read frame: {frame_name}")

  filtered_dec = [text for text in dec if text.strip()]

  if filtered_dec:
      most_frequent = Counter(filtered_dec).most_common(1)[0][0]
      print("decoded watermark:", most_frequent)
  else:
      print("No watermark decoded")




In [14]:
def initialize__System() :
  FrameCapture('/content/sample_data/foreman_cif.mp4')
  watermark=input('Enter watermark  ')
  key=input('Enter key  ')
  add_Watermark(watermark, '/content/foreman_cif',key)
  output_path = "/content/sample_data/output_video.avi"
  reconstruction('/content/foreman_cif', output_path)
  #extract('/content/foreman_cif',key)


In [16]:
initialize__System()

Enter watermark  CBIT_HYD
Enter key  abcd123
Average PSNR: 83.31369221218
Average SSIM: 0.9999994288158582
Average MSE : 0.00030497027567340067
watermark embedded


In [17]:
key=input('Enter key  ')
extract('/content/foreman_cif',key)

Enter key  abcd123
decoded watermark: CBIT_HYD
